### Random numbers

How important are random numbers when it comes to machine learning or deep learning? Well helluva important. How else are you going to initialize your parameters then? 

Actually a theoratical discussion on parameter initialization will hugely off track from the goal of these notebooks, so how about you read these wonderful pieces? (defintely later, because if you start reading them now and get lost you won't be able to get your interest back here).

1. [Initializing neural networks](https://www.deeplearning.ai/ai-notes/initialization/)
2. [Weight Initialization in Neural Networks: A Journey From the Basics to Kaiming](https://towardsdatascience.com/weight-initialization-in-neural-networks-a-journey-from-the-basics-to-kaiming-954fb9b47c79)
3. [Weight Initialization for Deep Learning Neural Networks](https://machinelearningmastery.com/weight-initialization-for-deep-learning-neural-networks/)

### Typical Random Numbers in python

In [2]:
import random

random.seed(42)
random.random()

0.6394267984578837

In [3]:
random.randrange(1, 10)

1

In [4]:
random.randrange(1, 10, 2)

5

### Random in Jax

Jax uses a different Pseudo Random Number Generator or, PRNG compared to the default in python or numpy. The primary goal behind Jax's implementaition was more to have a deterministic state for the PRNG among multiple devices (say GPUs), so that the results can be easily reproduced. 

Usually, in a distributed setup (with multiple devices) the state of a PRNG changes (even for the same seed), which can lead to inconsistent results between the devices. Ideally, you would want all your devices to sync the same PRNG configuration and produce random numbers the same way (i.e. without duplicates). This means that one, you need to parallely generate random numbers[2] while keeping the state of your PRNG consistent and two, you need a way to share the same state between your devices. Jax shares PRNG state via key splitting[1]. You can read more from the sources listed below.

**Sources:**
1. [Splittable pseudorandom number generators using cryptographic hashing](https://dl.acm.org/doi/10.1145/2578854.2503784)
2. [Parallel Random Numbers: As Easy as 1, 2, 3](https://www.thesalmons.org/john/random123/papers/random123sc11.pdf)

So in Jax, you generate a key for a random number, which is similar to what you call a seed for a random generator. (If anybody knows why 42 is a commonly used seed, kindly let me know!)

In [5]:
import jax as J

key = J.random.key(42) # holy 42!
key

Array((), dtype=key<fry>) overlaying:
[ 0 42]

In [6]:
J.random.normal(key)

Array(-0.02830462, dtype=float32)

In [7]:
J.random.normal(key)

Array(-0.02830462, dtype=float32)

#### Whoa there! why the same number again? 

Because it's the same seed? If you want your random numbers to be reproducible, you should always have the same seed. 

To get a new random number, you need a new key and for that, you just have to __split__ the key.


In [10]:
newkey, subkey = J.random.split(key)

In [11]:
J.random.normal(newkey)

Array(0.07592554, dtype=float32)

In [12]:
J.random.normal(newkey) # one more time .....

Array(0.07592554, dtype=float32)

### Multiple Splits

![what if](../images/what_if.jpg)


Off course! Legit question. You can mention how many keys you need with `split()`

In [13]:
newkey, *subkeys = J.random.split(key, 9)

In [14]:
for sk in subkeys:
    print(sk)

Array((), dtype=key<fry>) overlaying:
[  64467757 2916123636]
Array((), dtype=key<fry>) overlaying:
[2465931498  255383827]
Array((), dtype=key<fry>) overlaying:
[3134548294  894150801]
Array((), dtype=key<fry>) overlaying:
[2954079971 3276725750]
Array((), dtype=key<fry>) overlaying:
[2765691542  824333390]
Array((), dtype=key<fry>) overlaying:
[2768684296 3055579793]
Array((), dtype=key<fry>) overlaying:
[2547012911 1371500959]
Array((), dtype=key<fry>) overlaying:
[1016697191 2390192106]


In [15]:
for sk in subkeys:
    print(J.random.normal(sk))

0.60576403
0.4323065
-0.2818947
0.6549178
-0.2166012
-0.25440374
0.2886397
0.14384735


### Randomized Vectors

In [16]:
key = J.random.key(99) # why should 42 get all the attention?

In [17]:
a = J.random.normal(key, shape=(10, 1))
a

Array([[ 1.3088475 ],
       [-0.25337982],
       [-0.75657994],
       [ 0.7655394 ],
       [ 0.34965774],
       [ 1.6598269 ],
       [ 1.5128751 ],
       [-0.79486036],
       [-0.47534505],
       [-1.8532473 ]], dtype=float32)

In [18]:
mat_a = J.random.normal(key, shape=(6, 4))
mat_a

Array([[ 1.3088475 , -0.25337982, -0.75657994,  0.7655394 ],
       [ 0.34965774,  1.6598269 ,  1.5128751 , -0.79486036],
       [-0.47534505, -1.8532473 , -0.3786292 , -1.6755378 ],
       [-0.9703495 ,  0.6404915 ,  0.58516824, -0.1952716 ],
       [ 0.9720966 ,  0.3177939 ,  1.2931043 , -0.26136768],
       [-1.5360676 , -0.921658  , -1.0590659 , -1.4680037 ]],      dtype=float32)

### Arrays in Jax

If you're coming fresh out of a beginner python crash course, arrays are what you've known as lists, but a bit faster, better, more memory efficient etc. If you've experience working with numpy, Jax is just basic numpy with some magic sauce. Same applies for the arrays. Jax arrays are numpy arrays. 

In [19]:
# should we hard code again? nein!

import jax.numpy as jnp

b = jnp.arange(1, 10)
b

Array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

In [20]:
c = b * 2
c

Array([ 2,  4,  6,  8, 10, 12, 14, 16, 18], dtype=int32)

In [21]:
b / 2

Array([0.5, 1. , 1.5, 2. , 2.5, 3. , 3.5, 4. , 4.5], dtype=float32)

In [22]:
b + 2

Array([ 3,  4,  5,  6,  7,  8,  9, 10, 11], dtype=int32)

In [23]:
new_array = jnp.array(
    [b, c]
)
new_array

Array([[ 1,  2,  3,  4,  5,  6,  7,  8,  9],
       [ 2,  4,  6,  8, 10, 12, 14, 16, 18]], dtype=int32)

In [21]:
new_array.shape

(2, 9)

### Since they are like numpy arrays......

You should be able to change a value at a specific index, right? Erm............

In [24]:
b[2]

Array(3, dtype=int32)

In [25]:
b[2] = 99
b

TypeError: JAX arrays are immutable and do not support in-place item assignment. Instead of x[idx] = y, use x = x.at[idx].set(y) or another .at[] method: https://docs.jax.dev/en/latest/_autosummary/jax.numpy.ndarray.at.html

JAX arrays are immutable. Which means they can't be modified once created. But you may need to modify values at one point in your wonderful experiments with glorious functions called AI. Jax allows that but differently. You can modify the array but it'll create a new array and preserve the older one. 

In [26]:
b.at[2].set(99)

Array([ 1,  2, 99,  4,  5,  6,  7,  8,  9], dtype=int32)

In [27]:
# doesn't change!
b

Array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

But if I do this:

In [28]:
new_b = b.at[2].set(99)
new_b

Array([ 1,  2, 99,  4,  5,  6,  7,  8,  9], dtype=int32)

In [29]:
b

Array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

### More weirdo

```
b.at[100].set(100)
```

Do you think this line will run? b only has a length of 9. Whatever you say I won't blame you. You're neither wrong or correct in the JAX world.

In [30]:
b.at[100].set(100)

Array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=int32)

JAX completely ignores out of bound errors and will keep your original array intact. I don't know why it works this way when JAX imposes so much strict restrictions on other areas. So keep track of your indexes in your projects otherwise you'll be scratching your head why some value didn't update. 